## Problem Statement
Build a binary sentiment classifier on the IMDB dataset to predict whether a movie review is **positive (1)** or **negative (0)**.

## Objective
Implement and compare two sequence models:
- Simple RNN
- LSTM

## Evaluation Metric
Use **classification accuracy (%)** on the held-out IMDB test split as the primary metric.

**RNN**

In [12]:
!pip install datasets

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from collections import Counter
import numpy as np

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
dataset = load_dataset("imdb")

train_data = dataset['train']
test_data = dataset['test']

In [15]:
def build_vocab(texts, max_vocab=10000):
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())

    vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(max_vocab))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_data['text'])

In [16]:
MAX_LEN = 200

def encode_and_pad(text, vocab, max_len):
    tokens = [vocab.get(w, 1) for w in text.lower().split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    return tokens

In [17]:
x_train = torch.tensor([encode_and_pad(t, vocab, MAX_LEN) for t in train_data['text']])
y_train = torch.tensor(train_data['label']).float()

x_test = torch.tensor([encode_and_pad(t, vocab, MAX_LEN) for t in test_data['text']])
y_test = torch.tensor(test_data['label']).float()

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [18]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super(RNNModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [19]:
model = RNNModel(len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.round(torch.sigmoid(outputs))
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

Epoch 1, Loss: 0.6995, Accuracy: 50.35%
Epoch 2, Loss: 0.6892, Accuracy: 52.63%
Epoch 3, Loss: 0.6776, Accuracy: 54.76%


In [20]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.round(torch.sigmoid(outputs))

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy (RNN): {100*correct/total:.2f}%")

Test Accuracy (RNN): 53.71%


**LSTM** **Model**

In [21]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [22]:
model = LSTMModel(len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.round(torch.sigmoid(outputs))
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"LSTM Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

LSTM Epoch 1, Loss: 0.6934, Accuracy: 51.20%
LSTM Epoch 2, Loss: 0.6822, Accuracy: 54.21%
LSTM Epoch 3, Loss: 0.6410, Accuracy: 62.44%


In [23]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.round(torch.sigmoid(outputs))

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy (LSTM): {100*correct/total:.2f}%")

Test Accuracy (LSTM): 65.14%


## Results Summary
- RNN Test Accuracy: **53.71%**
- LSTM Test Accuracy: **65.14%**

The LSTM outperforms the simple RNN on this setup.